In [7]:
import os
import datetime
import argparse
from pathlib import Path
import multiprocessing as mp
from functools import partial
import yaml
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
# from src.module.utils import (
#     parse_args,
#     load_config,
#     load_npy,
#     save_npy,
#     multiprocess_function,
# )
# from src.module.prepare_dataset import preprocess, prepare

In [8]:
from dataclasses import dataclass


@dataclass
class Arguments:
    conf: str = "../../config/conf.yaml"
    strategy: str = "hypophetversion2"
    min = None
    max = None

In [ ]:
def set_environment(config):
    data_path = Path(config["data_path"]).expanduser()
    src_path = data_path / "mover" / "02.processed"
    save_path = data_path / "mover" / "05.dataset"
    file_path = sorted(src_path.rglob(f"*{config['fs']}fs*processed.h5"))
    mover_path = data_path / "mover" / "mover" / "mover-download.ics.uci.edu"
    cohort_path = (
        data_path
        / "mover"
        / "mover"
        / "mover-download.ics.uci.edu"
        / "EPIC_EMR"
        / "EMR"
    )
    env = {
        "src_path": src_path,
        "save_path": save_path,
        "file_path": file_path,
        "mover_path": mover_path,
        "cohort_path": cohort_path,
    }
    return env


def prepare_cohort(env) -> pd.DataFrame:
    pat_info = pd.read_csv(env["cohort_path"] / "patient_information.csv")
    pid_to_mrn = pd.read_csv(env["mover_path"] / "EPIC_MRN_PAT_ID.csv")
    cohort = (
        pd.merge(pid_to_mrn, pat_info, how="inner", on=["LOG_ID", "MRN"])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    # dept가 없어서 해당 처리는 하지 못하였다.
    cohort = (
        cohort.query("18 <= BIRTH_DATE < 65")
        .query("PRIMARY_ANES_TYPE_NM == 'General'")
        .reset_index(drop=True)
    )
    return cohort


def load_h5(file_path) -> tuple:
    with h5py.File(file_path, "r") as f:
        ts = f["signal/ts"][:]
        sig = f["signal/abp"][:]
        mbp = f["signal/mbp"][:]
        sqi = f["signal/sqi"][:]
        fs = f["signal"].attrs["fs"]
        sig_len = f["signal"].attrs["len"]
        cid = f.attrs["caseid"]

    return cid, fs, sig_len, ts, sig, mbp, sqi

In [29]:
args = Arguments()
config = load_config(args.conf)
env = set_environment(config)

In [30]:
cohort = prepare_cohort(env)

In [35]:
cohort_caseid = cohort.PAT_ID.values
directories = np.array(os.listdir(env["src_path"]))
cohort_dirs = directories[np.isin(directories, cohort_caseid)]
# cohort_file_path = list(
#     filter(lambda x: str(x).split("/")[-2] in cohort_dirs, env["file_path"])
# )
cohort_file_path = list(
    filter(lambda x: str(x).split("\\")[-2] in cohort_dirs, env["file_path"])
)

In [36]:
cid, _, _, _, _, _, _ = load_h5(cohort_file_path[1])

In [37]:
str(cid, "utf-8")

'01611c309d898918'

In [38]:
cohort_file_path[1]

WindowsPath('D:/Github/hypotension-classifier/data/mover/02.processed/01611c309d898918/01611c309d898918_100fs_processed.h5')

In [39]:
pat_info = pd.read_csv(
    "../../data/mover/mover/mover-download.ics.uci.edu/EPIC_EMR/EMR/patient_information.csv"
)
pid_to_mrn = pd.read_csv(
    "../../data/mover/mover/mover-download.ics.uci.edu/EPIC_MRN_PAT_ID.csv"
)

In [40]:
cohort = (
    pd.merge(pid_to_mrn, pat_info, how="inner", on=["LOG_ID", "MRN"])
    .drop_duplicates()
    .reset_index(drop=True)
)

,BIRTH_DATE,HEIGHT,WEIGHT,SEX,PRIMARY_ANES_TYPE_NM,ASA_RATING_C,ASA_RATING,PATIENT_CLASS_GROUP,PATIENT_CLASS_NM,PRIMARY_PROCEDURE_NM,IN_OR_DTTM,OUT_OR_DTTM,AN_START_DATETIME,AN_STOP_DATETIME,SOURCE_KEY,SOURCE_NAME,NAME,REF_BILL_CODE_SET_NAME,REF_BILL_CODE
0,47,NaN,2832.47,Female,General,3.0,Severe Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PUBOVAGINAL SLING, WITH CYSTOSCOPY",12/20/18 12:27,12/20/18 17:25,12/20/18 12:27,12/20/18 17:34,3,Final Diagnosis Primary Code Set,Inflammatory disease of cervix uteri,ICD-10-CM,N72
1,47,NaN,2832.47,Female,General,3.0,Severe Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PUBOVAGINAL SLING, WITH CYSTOSCOPY",12/20/18 12:27,12/20/18 17:25,12/20/18 12:27,12/20/18 17:34,11,ICD Procedure Primary Code Set,"Reposition Urethra, Open Approach",ICD-10-PCS,0TSD0ZZ
2,47,NaN,2832.47,Female,General,3.0,Severe Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PUBOVAGINAL SLING, WITH CYSTOSCOPY",12/20/18 12:27,12/20/18 17:25,12/20/18 12:27,12/20/18 17:34,11,ICD Procedure Primary Code Set,"Removal of Contracep Dev from Uterus & Cervix,...",ICD-10-PCS,0UPD0HZ
3,47,NaN,2832.47,Female,General,3.0,Severe Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PUBOVAGINAL SLING, WITH CYSTOSCOPY",12/20/18 12:27,12/20/18 17:25,12/20/18 12:27,12/20/18 17:34,21,Charge CPT Code,MEDICAL/SURGICAL SUPPLIES AND DEVICES - OTHER ...,NaN,C1771
4,47,NaN,2832.47,Female,General,3.0,Severe Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PUBOVAGINAL SLING, WITH CYSTOSCOPY",12/20/18 12:27,12/20/18 17:25,12/20/18 12:27,12/20/18 17:34,21,Charge CPT Code,PHARMACY - EXTENSION OF 025X - DRUGS REQUIRING...,NaN,A4217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2629013,65,NaN,2933.00,Male,General,2.0,Mild Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PENILE PROSTHESIS",8/22/21 10:10,8/22/21 12:35,8/22/21 10:10,8/22/21 12:39,21,Charge CPT Code,EMERGENCY ROOM - GENERAL CLASSIFICATION,NaN,96375
2629014,65,NaN,2933.00,Male,General,2.0,Mild Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PENILE PROSTHESIS",8/22/21 10:10,8/22/21 12:35,8/22/21 10:10,8/22/21 12:39,21,Charge CPT Code,EMERGENCY ROOM - GENERAL CLASSIFICATION,NaN,99285
2629015,65,NaN,2933.00,Male,General,2.0,Mild Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PENILE PROSTHESIS",8/22/21 10:10,8/22/21 12:35,8/22/21 10:10,8/22/21 12:39,21,Charge CPT Code,PHARMACY - EXTENSION OF 025X - DRUGS REQUIRING...,NaN,J0360
2629016,65,NaN,2933.00,Male,General,2.0,Mild Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PENILE PROSTHESIS",8/22/21 10:10,8/22/21 12:35,8/22/21 10:10,8/22/21 12:39,21,Charge CPT Code,PHARMACY - EXTENSION OF 025X - DRUGS REQUIRING...,NaN,J7120


In [42]:
cohort.query("18 <= BIRTH_DATE < 65").query("PRIMARY_ANES_TYPE_NM == 'General'").PAT_ID.nunique()

23753

In [ ]:
(pat_info.query("18 <= BIRTH_DATE < 65").drop_duplicates())

In [9]:
pat_info.query("PRIMARY_ANES_TYPE_NM == 'General'")

,LOG_ID,MRN,DISCH_DISP_C,DISCH_DISP,HOSP_ADMSN_TIME,HOSP_DISCH_TIME,LOS,ICU_ADMIN_FLAG,SURGERY_DATE,BIRTH_DATE,...,PRIMARY_ANES_TYPE_NM,ASA_RATING_C,ASA_RATING,PATIENT_CLASS_GROUP,PATIENT_CLASS_NM,PRIMARY_PROCEDURE_NM,IN_OR_DTTM,OUT_OR_DTTM,AN_START_DATETIME,AN_STOP_DATETIME
0,cd1636c6279d73a0,2e92522a5f2a2a22,15.0,Home Routine,12/20/18 11:20,12/21/18 12:15,1.0,No,12/20/18 0:00,47,...,General,3.0,Severe Systemic Disease,Outpatient,Hospital Outpatient Surgery,"INSERTION, PUBOVAGINAL SLING, WITH CYSTOSCOPY",12/20/18 12:27,12/20/18 17:25,12/20/18 12:27,12/20/18 17:34
1,fc53c06ee2cfe438,b7a91e623d957d8f,16.0,Hospice Facility,9/27/19 9:02,10/18/19 18:13,21.0,Yes,9/27/19 0:00,81,...,General,3.0,Severe Systemic Disease,Outpatient,Hospital Outpatient Surgery,GI EGD DILATION,9/27/19 9:52,9/27/19 11:27,9/27/19 9:52,9/27/19 11:44
2,2f0090c47da34608,b68d6c7198f30f73,15.0,Home Routine,4/8/19 9:33,4/11/19 12:34,3.0,Yes,4/8/19 0:00,49,...,General,2.0,Mild Systemic Disease,Inpatient,Hospital Inpatient Surgery,CRANIECTOMY,4/8/19 14:41,4/8/19 18:46,4/8/19 14:41,4/8/19 18:53
3,6d3b621880c6ced4,46aa48e89b9c50e8,15.0,Home Routine,12/3/18 5:57,12/4/18 12:57,1.0,No,12/3/18 0:00,54,...,General,2.0,Mild Systemic Disease,Outpatient,Hospital Outpatient Surgery,"REVISION, RECONSTRUCTION, BREAST",12/3/18 8:09,12/3/18 11:49,12/3/18 8:09,12/3/18 11:52
4,82f89e10f2855420,c2756216cc89b795,15.0,Home Routine,4/19/19 6:19,4/20/19 9:26,1.0,No,4/19/19 0:00,44,...,General,2.0,Mild Systemic Disease,Outpatient,Hospital Outpatient Surgery,"REVISION, RECONSTRUCTION, BREAST",4/19/19 8:13,4/19/19 14:51,4/19/19 8:14,4/19/19 15:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65723,b619db184a3f7d86,aecf54e7e0077c0b,6.0,Skilled Nursing Facility,6/7/21 3:35,8/3/21 13:00,57.0,Yes,6/19/21 0:00,65,...,General,3.0,Severe Systemic Disease,Inpatient,Inpatient Admission,"EGD, WITH GASTROJEJUNOSTOMY TUBE INSERTION, PE...",6/19/21 7:12,6/19/21 8:22,6/19/21 7:12,6/19/21 8:27
65724,54b5138bdecfd66f,cd3f8576771bf9d2,15.0,Home Routine,11/11/21 13:14,11/11/21 20:14,0.0,No,11/11/21 0:00,48,...,General,2.0,Mild Systemic Disease,Outpatient,Hospital Outpatient Surgery,"HYSTEROSCOPY, WITH DILATION AND CURETTAGE OF U...",11/11/21 15:43,11/11/21 17:47,11/11/21 15:43,11/11/21 17:54
65725,35d4b39cdc9525ee,50b9fdfb13f2d8ec,15.0,Home Routine,3/7/22 11:24,3/8/22 11:30,1.0,Yes,3/7/22 0:00,62,...,General,3.0,Severe Systemic Disease,Outpatient,Hospital Outpatient Surgery,FOOT AND ANKLE - MASS OR FOREIGN BODY EXCISION,3/7/22 17:15,3/7/22 20:26,3/7/22 17:15,3/7/22 20:32
65726,399d457014a50d02,89d24047a41d7048,15.0,Home Routine,9/6/22 10:25,9/6/22 22:50,0.0,No,9/6/22 0:00,34,...,General,2.0,Mild Systemic Disease,Inpatient,Inpatient Admission,"CYSTOSCOPY, WITH URETERAL STENT INSERTION",9/6/22 20:54,9/6/22 21:49,9/6/22 20:54,9/6/22 21:53
